# 测试集

In [1]:
import json

image_root = "/mnt/petrelfs/share_data/liqingyun/data/mmrotate_data"
annotation_path = "/mnt/petrelfs/liqingyun/share_data/data/mmrotate_data/LRS_VQA/LRS_VQA_merged.jsonl"

with open(annotation_path, "r") as f:
    data = [json.loads(line) for line in f.readlines()]

data[0], len(data)

({'question_id': 'FAIR_1',
  'image': 'LRS_VQA/image/15565.tif',
  'category': 'reasoning',
  'text': 'Is the bottom-most and largest bridge crossing a river?\n Answer the question using a single word or phrase.',
  'ground_truth': 'yes',
  'hbox': [282, 3418, 763, 4050],
  'rbox': [282.0, 4013.0, 334.0, 4050.0, 763.0, 3456.0, 713.0, 3418.0],
  'abs_size': 'large',
  'image_size': [4000, 4000],
  'size_bin': '256x256'},
 7333)

In [2]:
image2data = {}
for sample in data:
    if sample['image'] not in image2data:
        image2data[sample['image']] = []
    image2data[sample['image']].append(sample)

print(f"total images: {len(image2data)}")


total images: 1228


### 查看每个任务的response diversity

In [3]:
task_data = {}
for item in data:
    task = item['category']
    if task not in task_data:
        task_data[task] = []
    task_data[task].append(item)

for task, task_items in task_data.items():
    responses = set()
    for item in task_items:
        responses.add(item['ground_truth'])
    print(f"{task} task outputs: len(task_items)={len(task_items)}  len(responses)={len(responses)}")
    if task in ['rural or urban', 'object color']:
        print(responses)
    elif task == 'count':
        print(sorted(responses, key=lambda x: int(x)))

reasoning task outputs: len(task_items)=1000  len(responses)=465
object shape task outputs: len(task_items)=885  len(responses)=78
rural or urban task outputs: len(task_items)=1252  len(responses)=11
{'undetermined', 'no', 'undeterminable', 'grid layout', 'yes', 'intersection', 'urban', 'rural', 'indeterminable', 'it is ambiguous', 'industrial'}
object category task outputs: len(task_items)=987  len(responses)=337
object status task outputs: len(task_items)=1000  len(responses)=392
count task outputs: len(task_items)=1199  len(responses)=20
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']
object color task outputs: len(task_items)=765  len(responses)=42
{'yellow', 'dark grey', 'white and grey', 'orange', 'blue and white', '3', 'dark gray', 'rust', 'grey', 'blue', 'cream', 'various', 'dark blue', 'no', '4', 'red', 'dark green', 'brown', 'greenish', 'black', 'gray', 'building', 'can&rsquo;t determine', 'beige', 'turquoise', '

# 训练集

### 第一次获取的

似乎是LRS-VQA论文里用来做训练实验的？9213个样本 4795张图 来自dota2.0的trainval，GLH-Bridge的trainval，和STAR

Combine_vqa_pairs_filter_multi_turn_9k.json

但是直接是 image 和 conversations，没有box信息和type信息

对话轮数很多

 {2: 1478, 4: 1483, 6: 819, 8: 466, 10: 310, 12: 4657}
 
按轮次算一共 38257 轮对话，尽管是多轮对话，还是有不同样本公用图的情况，估计是6轮是上限，到6轮截断

其中 24638 是数字形式的respnse, 13619 是yes/no形式的response

没存，但是模板造的，直接拿目标检测的标签造的

In [4]:
# 读取数据
from rscovlm.eval.data.data import load_json_or_jsonl
annotation_path = "/mnt/petrelfs/liqingyun/share_data/data/mmrotate_data/LRS_VQA/Combine_vqa_pairs_filter_multi_turn_9k.json"
data = load_json_or_jsonl(annotation_path)
data[0], len(data)

/mnt/petrelfs/liqingyun/miniconda3/envs/rscovlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/petrelfs/liqingyun/miniconda3/envs/rscovlm/lib/python3.10/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


[2025-05-21 14:28:28,978] [WARNING] [real_accelerator.py:194:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2025-05-21 14:28:28,985] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cpu (auto detect)


/mnt/petrelfs/liqingyun/miniconda3/envs/rscovlm/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


({'id': '000000000001_0',
  'image': 'RS_ObjectDetection/DOTA2.0/train/images/P0000.png',
  'conversations': [{'from': 'human',
    'value': '<image>\nWhat is the number of plane in the center part of the image? \nAnswer the question using a single word or phrase.'},
   {'from': 'gpt', 'value': '1'}]},
 9213)

In [5]:
# 查看有多少张图，判断重合率
image2data = {}
for sample in data:
    if sample['image'] not in image2data:
        image2data[sample['image']] = []
    image2data[sample['image']].append(sample)

print(f"total images: {len(image2data)}")

total images: 4795


In [6]:
# 检查是不是所有的图都下载好了
import os
import json
from pprint import pprint
root = '/mnt/petrelfs/share_data/liqingyun/datasets/mmrotate_data/LRS_VQA'
subsets = {}
for item in data:
    folder = os.path.dirname(item['image'])
    image_path = os.path.join(root, item['image'])
    if not os.path.exists(image_path):
        print(f"image not exists: {image_path}")
    if folder not in subsets:
        subsets[folder] = []
    subsets[folder].append(item)
print(f"total folders: {len(subsets)}: ", subsets.keys())

for folder, items in subsets.items():
    print(f"{folder}: {len(items)} {json.dumps(items[0], indent=4)}")

total folders: 4:  dict_keys(['RS_ObjectDetection/DOTA2.0/train/images', 'RS_ObjectDetection/DOTA2.0/val/images', 'RS_ObjectDetection/GLH-Bridge/trainval_images', 'STAR/ori_images'])
RS_ObjectDetection/DOTA2.0/train/images: 1090 {
    "id": "000000000001_0",
    "image": "RS_ObjectDetection/DOTA2.0/train/images/P0000.png",
    "conversations": [
        {
            "from": "human",
            "value": "<image>\nWhat is the number of plane in the center part of the image? \nAnswer the question using a single word or phrase."
        },
        {
            "from": "gpt",
            "value": "1"
        }
    ]
}
RS_ObjectDetection/DOTA2.0/val/images: 340 {
    "id": "000000001091_0",
    "image": "RS_ObjectDetection/DOTA2.0/val/images/P0004.png",
    "conversations": [
        {
            "from": "human",
            "value": "<image>\nHow many small-vehicles are there in the top-right part of the image? \nAnswer the question using a single word or phrase."
        },
        {
 

In [7]:
# 大致判断下数据的对话轮数、任务类型
num_rounds = {}
n_digit = 0
n_yesno = 0
for item in data:
    n_rounds = len(item['conversations'])
    if n_rounds not in num_rounds:
        num_rounds[n_rounds] = 0
    num_rounds[n_rounds] += 1

    for i in range(0, n_rounds, 2):
        if i + 1 >= n_rounds:
            break
        question = item['conversations'][i]['value']
        response = item['conversations'][i + 1]['value']

        if response.isdigit():
            n_digit += 1
        else:
            assert response.lower() in ['yes', 'no'], f"invalid response: {response}"
            n_yesno += 1

num_rounds, n_digit, n_yesno, sum(int(k / 2) * v for k, v in num_rounds.items())

({2: 1478, 4: 1483, 6: 819, 12: 4657, 10: 310, 8: 466}, 24638, 13619, 38257)

### 第二次获取的

这回的样本超级多，足足有 178905 个，但其实是单轮对话，但也比之前多了，图仅有 5232 略多于第一批

其中 19977 是数字形式的respnse（少于之前）, 75697 是yes/no形式的response（多于之前），剩下83231是其他形式

但是和第一次的似乎不是包含关系

但是带了很多附属产品：
在院线的metadata下带了

['image_name', 'image_width', 'image_height', 'hbox', 'rbox', 'crop_hbox']

GPT4V造的和模板造的混在一起了，主要是GPT4V造的

In [27]:
# 读取数据
import json

files = [
    '../data/LRS_VQA/Mix_LS-VQA_DOTA2-Bridge-STAR-179k_add_box_part1.json', 
    '../data/LRS_VQA/Mix_LS-VQA_DOTA2-Bridge-STAR-179k_add_box_part2.json',
    '../data/LRS_VQA/Mix_LS-VQA_DOTA2-Bridge-STAR-179k_add_box_part3.json',
    '../data/LRS_VQA/Mix_LS-VQA_DOTA2-Bridge-STAR-179k_add_box_part4.json'
]

data2 = []
for file in files:
    with open(file, 'r') as f:
        data2.extend(json.load(f))

num_rounds = {}
n_digit = 0
n_yesno = 0
invalid = []
for item in data2:
    n_rounds = len(item['conversations'])
    if n_rounds not in num_rounds:
        num_rounds[n_rounds] = 0
    num_rounds[n_rounds] += 1

    for i in range(0, n_rounds, 2):
        if i + 1 >= n_rounds:
            break
        question = item['conversations'][i]['value']
        response = item['conversations'][i + 1]['value']

        if response.isdigit():
            n_digit += 1
        elif response.lower() in ['yes', 'no']:
            n_yesno += 1
        else:
            invalid.append(item)

print(data2[0].keys())
print(data2[0]['bbox_info'].keys())
print(json.dumps(data2[0], indent=4))
num_rounds, n_digit, n_yesno, len(invalid), sum(int(k / 2) * v for k, v in num_rounds.items())

dict_keys(['id', 'image', 'bbox_info', 'conversations'])
dict_keys(['image_name', 'image_width', 'image_height', 'hbox', 'rbox', 'crop_hbox'])
{
    "id": "DOTAv2_train_wh1400_P3397-3",
    "image": "RS_ObjectDetection/DOTA2.0/train/images/P3397.png",
    "bbox_info": {
        "image_name": "P3397.png",
        "image_width": 4096,
        "image_height": 4096,
        "hbox": [
            1153,
            2568,
            1201,
            2596
        ],
        "rbox": [
            1153.0,
            2568.0,
            1201.0,
            2568.0,
            1201.0,
            2596.0,
            1153.0,
            2596.0
        ],
        "crop_hbox": [
            953,
            2368,
            1401,
            2796
        ]
    },
    "conversations": [
        {
            "from": "human",
            "value": "<image>\nIs the area surrounding the bottom-most and the smaller roundabout urban?\n Answer the question using a single word or phrase."
        },
     

({2: 178905}, 19977, 75697, 83231, 178905)

In [9]:
# 查看有多少张图，判断重合率
image2data2 = {}
for sample in data2:
    if sample['image'] not in image2data2:
        image2data2[sample['image']] = {}
    image2data2[sample['image']][sample['conversations'][0]['value']] = sample

print(f"total images: {len(image2data2)}")

total images: 5232


In [18]:
# 试着和第一次的数据对应，发现对应不上
example = data[0]
image = example['image']
query = example['conversations'][0]['value']

print(query); print()

for q in image2data2[image].keys():
    print(q)

<image>
What is the number of plane in the center part of the image? 
Answer the question using a single word or phrase.

<image>
Does the storage-tank appear surrounded by roads?
 Answer the question using a single word or phrase.
<image>
Is the storage-tank located in an urban area?
 Answer the question using a single word or phrase.
<image>
Are there any buildings adjacent to the storage-tank?
 Answer the question using a single word or phrase.
<image>
Is the storage-tank circular in shape?
 Answer the question using a single word or phrase.


### 第三次获取的

第一次没存box的，并且稍微放宽规则让样本变多

In [11]:
import json

path_list = [
    '../data/LRS_VQA/template_vqa_3datasets_withbox/DOTAv2_train_vqa_pairs.jsonl', 
    '../data/LRS_VQA/template_vqa_3datasets_withbox/DOTAv2_val_vqa_pairs.jsonl',
    '../data/LRS_VQA/template_vqa_3datasets_withbox/GLH-Bridge_trainval_vqa_pairs.jsonl',
    '../data/LRS_VQA/template_vqa_3datasets_withbox/STAR_trainval_vqa_pairs.jsonl',
]

types = set()
for p in path_list:
    with open(p, 'r') as f:
        data3 = [json.loads(line) for line in f.readlines()]
    print(f"{p}: {len(data4)}")
    print(data4[0].keys())
    image2data4 = {}
    for sample in data3:
        if sample['image'] not in image2data3:
            image2data3[sample['image']] = []
        image2data3[sample['image']].append(sample)
        types.add(sample['type'])
    print(f"total images: {len(image2data3)}")

print(types)
print(json.dumps(next(iter(item for item in data3 if item['type'] == 'count'), None), indent=4))
print(json.dumps(next(iter(item for item in data3 if item['type'] == 'compare'), None), indent=4))

../data/LRS_VQA/template_vqa_3datasets_withbox/DOTAv2_train_vqa_pairs.jsonl: 17443
dict_keys(['id', 'image', 'image_path', 'type', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'box_info'])
total images: 1816
../data/LRS_VQA/template_vqa_3datasets_withbox/DOTAv2_val_vqa_pairs.jsonl: 5699
dict_keys(['id', 'image', 'image_path', 'type', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'box_info'])
total images: 592
../data/LRS_VQA/template_vqa_3datasets_withbox/GLH-Bridge_trainval_vqa_pairs.jsonl: 9669
dict_keys(['id', 'image', 'image_path', 'type', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'box_info'])
total images: 3502
../data/LRS_VQA/template_vqa_3datasets_withbox/STAR_trainval_vqa_pairs.jsonl: 26763
dict_keys(['id', 'image', 'image_path', 'type', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'box_info'])
total images: 1009
{'compare', 'count'}
{
    "id": "0000_qs_1",
    "image": "0000.png",
    "image_path": "/gruntdata/

### 第四次获取的

In [9]:
import json

path_list = [
    '../data/LRS_VQA/5_22_GPT4V_LS-VQA_DOTA2-Bridge-STAR_with_boxx_info_159k_part_0.json',
    '../data/LRS_VQA/5_22_GPT4V_LS-VQA_DOTA2-Bridge-STAR_with_boxx_info_159k_part_1.json',
    # '../data/LRS_VQA/5_22_template_DOTA-Bridge-STAR_with_boxx_info_60k_part_0.json',
    # '../data/LRS_VQA/5_22_template_DOTA-Bridge-STAR_with_boxx_info_60k_part_1.json',
    # '../data/LRS_VQA/5_22_template_DOTA-Bridge-STAR_with_boxx_info_60k_part_2.json',
    # '../data/LRS_VQA/5_22_template_DOTA-Bridge-STAR_with_boxx_info_60k_part_3.json',
    # '../data/LRS_VQA/5_22_template_DOTA-Bridge-STAR_with_boxx_info_60k_part_4.json',
]

for p in path_list:
    with open(p, 'r') as f:
        data4 = json.load(f)
    print(f"{p}: {len(data4)}")
    print(data4[0].keys())
    image2data4 = {}
    for sample in data4:
        if sample['image'] not in image2data4:
            image2data4[sample['image']] = []
        image2data4[sample['image']].append(sample)
    print(f"total images: {len(image2data4)}")

data4[0], data4[0]['bbox_info'].keys(), len(data4[0]['bbox_info']['crop_hbox'])

../data/LRS_VQA/5_22_GPT4V_LS-VQA_DOTA2-Bridge-STAR_with_boxx_info_159k_part_0.json: 79353
dict_keys(['id', 'image', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'bbox_info'])
total images: 5223
../data/LRS_VQA/5_22_GPT4V_LS-VQA_DOTA2-Bridge-STAR_with_boxx_info_159k_part_1.json: 79352
dict_keys(['id', 'image', 'image_ori_width', 'image_ori_height', 'question', 'answer', 'bbox_info'])
total images: 5214


({'id': 'DOTAv2_train_wh1400_P1521-6',
  'image': 'RS_ObjectDetection/DOTA2.0/train/images/P1521.png',
  'image_ori_width': 4000,
  'image_ori_height': 4000,
  'question': 'Does the top-most ground-track-field have lights around it?\n Answer the question using a single word or phrase.',
  'answer': 'yes',
  'bbox_info': {'rbox': [[1238.0,
     841.0,
     1296.0,
     829.0,
     1319.0,
     1002.0,
     1257.0,
     1013.0]],
   'crop_hbox': [[1038, 629, 1519, 1213]]}},
 dict_keys(['rbox', 'crop_hbox']),
 1)